In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data & Index Date
df = pd.read_csv("financial_regression.csv")
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# 2. Clean Macro Economic Columns & Forward Fill
cols_to_drop = ['GDP', 'us_rates_%', 'CPI']
df = df.drop(columns=cols_to_drop).ffill().dropna()

# 3. Create Target Variable: Daily Gold Price Change (Durağanlaştırma)
df['gold_change'] = df['gold close'].diff()

# Remove gold columns to prevent data leakage
features_to_drop = ['gold open', 'gold high', 'gold low', 'gold close', 'gold volume', 'gold_change']
X = df.drop(columns=features_to_drop).iloc[1:]
y = df['gold_change'].iloc[1:]

In [31]:
from sklearn.model_selection import train_test_split

# %80 Training, %20 Test (Chronological order preserving)
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

In [32]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Hyperparameter search space
param_grid = {
    'max_depth': [2, 3, 4, 5, 6],
    'min_samples_split': [10, 20, 50],
    'min_samples_leaf': [5, 10, 20]
}

dt = DecisionTreeRegressor(random_state=42)
grid_search = GridSearchCV(dt, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

In [33]:
# 1. Daily Difference Predictions
test_pred_change = best_model.predict(X_test)

# 2. Reconstruct Real Price: Today's Price = Yesterday's Real Price + Predicted Change
y_test_real = df['gold close'].iloc[train_size+1:]
y_test_prev = df['gold close'].iloc[train_size:-1].values
y_test_pred_price = y_test_prev + test_pred_change

# Metrics
real_price_mae = mean_absolute_error(y_test_real, y_test_pred_price)
real_price_r2 = r2_score(y_test_real, y_test_pred_price)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Reconstructed Real Price MAE: {real_price_mae:.2f} $")
print(f"Reconstructed Real Price R2 Score: {real_price_r2:.4f}")

Best Hyperparameters: {'max_depth': 2, 'min_samples_leaf': 5, 'min_samples_split': 20}
Reconstructed Real Price MAE: 1.23 $
Reconstructed Real Price R2 Score: 0.9943


In [34]:
#In this project, we built a gold price prediction model using a Kaggle dataset and a DecisionTreeRegressor. We split the data chronologically with TimeSeriesSplit to avoid data leakage caused by random shuffling. Since decision trees cannot predict new trends outside past values, we converted raw prices into daily price changes to make the data stationary. The final reconstructed prices achieved a high $R^2$ score of 0.9943, but this is due to high day-to-day correlation rather than model strength. The model achieved a MAE of $1.23, showing that evaluating daily changes is much more meaningful than looking at raw price levels. 